# Chapter 23 - Beyond the Transformer: Benchmarks (live)

Companion notebook for [Chapter 23](../part-6-frontier/23-alternative-architectures.md). The case for sub-quadratic architectures is quantitative, so let's *measure* it: attention's $O(n^2)$ cost vs linear, the KV cache that grows vs a fixed SSM state, and the SSM recurrence/convolution **duality** plus Mamba-style **selectivity**.

> Pure NumPy + matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
np.random.seed(0)

## 1. Quadratic vs linear cost

Attention does $O(n^2 d)$ work; a linear/SSM layer does $O(n d^2)$ - linear in sequence length. The analytical curves diverge fast, and the measured wall-time of (vectorized) softmax attention grows quadratically.

In [ ]:
d = 64
ns = np.array([64, 128, 256, 512, 1024, 2048])
quad = ns ** 2 * d
lin = ns * d ** 2

def softmax_attn(Q, K, V):
    s = Q @ K.T / np.sqrt(Q.shape[1])
    s = np.tril(np.exp(s - s.max(1, keepdims=True)))
    return (s / s.sum(1, keepdims=True)) @ V

times = []
for n in ns:
    Q, K, V = (np.random.randn(n, d) for _ in range(3))
    t0 = time.time(); softmax_attn(Q, K, V); times.append(time.time() - t0)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(ns, quad, 'o-', label='attention  ~ n^2 d')
ax[0].plot(ns, lin, 's-', label='linear/SSM ~ n d^2')
ax[0].set_xlabel('sequence length n'); ax[0].set_ylabel('FLOPs (relative)')
ax[0].set_title('Analytical cost'); ax[0].legend()
ax[1].loglog(ns, times, 'o-')
ax[1].set_xlabel('sequence length n'); ax[1].set_ylabel('seconds')
ax[1].set_title('Measured softmax-attention time (log-log)')
plt.tight_layout(); plt.show()

## 2. Memory: growing KV cache vs fixed SSM state

A transformer caches K and V for **every** past token, so memory grows with context. An SSM/linear model keeps a **fixed-size** state regardless of length - the core inference advantage for long context.

In [ ]:
def kv_bytes(t, n_layers=32, n_kv_heads=8, d_head=128, dtype=2):
    return t * n_layers * n_kv_heads * d_head * 2 * dtype      # K and V, grows with t

def ssm_bytes(t, n_layers=32, d_state=16, d_model=2048, dtype=2):
    return np.full_like(np.asarray(t, float), n_layers * d_state * d_model * dtype)

ctx = np.array([128, 512, 2048, 8192, 32768, 131072])
plt.figure(figsize=(5.5, 3.8))
plt.loglog(ctx, kv_bytes(ctx) / 1e6, 'o-', label='transformer KV cache')
plt.loglog(ctx, ssm_bytes(ctx) / 1e6, 's-', label='SSM state (fixed)')
plt.xlabel('context length (tokens)'); plt.ylabel('memory (MB)')
plt.title('KV cache grows; SSM state is flat'); plt.legend()
plt.tight_layout(); plt.show()

## 3. The SSM duality: recurrence == convolution

A time-invariant SSM can be run as a step-by-step recurrence (cheap inference, constant memory) **or** as a convolution with a fixed kernel (parallel training). They give the *same* output - the trick that makes SSMs trainable like CNNs and deployable like RNNs.

In [ ]:
def ssm_recurrent(x, Abar, Bbar, C):
    h = np.zeros_like(Abar); y = np.zeros(len(x))
    for t in range(len(x)):
        h = Abar * h + Bbar * x[t]
        y[t] = C @ h
    return y

def ssm_conv(x, Abar, Bbar, C):
    L = len(x)
    K = np.array([C @ (Abar ** k * Bbar) for k in range(L)])   # kernel C Abar^k Bbar
    return np.convolve(x, K)[:L]

L = 60
Abar = np.array([0.9, 0.7, 0.5]); Bbar = np.array([1.0, 0.5, 0.2]); C = np.ones(3)
x = np.random.randn(L)
y_rec, y_cnv = ssm_recurrent(x, Abar, Bbar, C), ssm_conv(x, Abar, Bbar, C)
print('recurrent == convolution:', np.allclose(y_rec, y_cnv))

kernel = np.array([C @ (Abar ** k * Bbar) for k in range(L)])
plt.figure(figsize=(5.5, 3.2))
plt.stem(kernel[:20])
plt.title('SSM impulse-response kernel (sum of decaying modes)')
plt.xlabel('lag k'); plt.ylabel('kernel weight'); plt.tight_layout(); plt.show()

## 4. Mamba's selectivity: input-dependent dynamics

S4 uses the *same* dynamics for every token. Mamba makes the step size $\Delta$ (and B, C) **input-dependent**, so it can choose how fast to update its state. Here the same impulse decays at different rates depending on $\Delta$ - the knob that gives content-based memory.

In [ ]:
def selective_ssm(x, A, B, C, delta):
    N = A.shape[0]; h = np.zeros(N); y = np.zeros(len(x))
    for t in range(len(x)):
        h = np.exp(delta[t] * A) * h + (delta[t] * B[t]) * x[t]
        y[t] = C[t] @ h
    return y

L, N = 40, 4
A = -np.array([0.2, 0.5, 1.0, 2.0])
B = np.ones((L, N)); C = np.ones((L, N))
x = np.zeros(L); x[0] = 1.0                          # an impulse

plt.figure(figsize=(5.5, 3.3))
for dval in (0.2, 0.5, 1.0):
    y = selective_ssm(x, A, B, C, np.full(L, dval))
    plt.plot(y, label=f'delta = {dval}')
plt.title('Selective SSM: larger delta -> faster state update / decay')
plt.xlabel('time step'); plt.ylabel('output'); plt.legend(); plt.tight_layout(); plt.show()

## Takeaway

The transformer's costs are real and measurable: quadratic compute and a linearly growing KV cache. SSMs and linear attention trade a little recall for **linear time** and **constant memory**, and the recurrence/convolution duality plus Mamba's selectivity are what make them both trainable and competitive. Hybrids keep a few attention layers to restore exact recall - the current practical sweet spot.

[Back to Chapter 23](../part-6-frontier/23-alternative-architectures.md) | [Solutions](../solutions/23-alternative-architectures-solutions.md)